# illust-asset-forge

上から順に実行する。2 つ目のコードセル（モデルの読み込み）は数分かかる。

## 1. 準備（リポジトリ・ライブラリ）

Drive をつなぎ、リポジトリを取り、ライブラリを入れる。Claude が設定を直して push した後は、**このセルだけ**を実行し直せば取り込める（モデルは読み込み直さない）。

In [ ]:
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")

REPO = "/content/illust-asset-forge"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/akabee0161/illust-asset-forge.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

!pip install -q -r {REPO}/requirements-colab.txt
!pip install -q -e {REPO} --no-deps

os.chdir(REPO)
# pip install -e は、動いているカーネルの import には効かない。src を直接足す
if os.path.join(REPO, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO, "src"))
# 実行し直したときに古いコードのままにならないよう、読み込み済みの artgen を捨てる
for name in [m for m in sys.modules if m == "artgen" or m.startswith("artgen.")]:
    del sys.modules[name]
from artgen import forge as F

## 2. モデルの読み込み

`forge` が無いときだけ読み込む（2 回目以降は何もしない）。メモリ不足で止まったときだけ、`F.load(".", low_memory=True)` にする（遅くなる）。

In [ ]:
if "forge" not in globals():
    forge = F.load(".")

## 3. 顔（通常）

1 枚できるたびに表示される。調整中は `n=2` など少なくし、気に入らなければ ■ で止めて、パラメータを変えて作り直す（止めても、できた分は Drive に残る）。一気に作るときは `n=16` などにして、後で Drive から見る。選んだファイル名は次のセルで使う。

In [ ]:
paths = forge.generate("ally05", "face", n=8, on_image=F.show_one)

## 4. 表情

採用した顔を `source` に渡す。`inpaint_box` が要るので、顔を採用して Claude が書いて push した後に、1 のセルを実行し直して設定を取り込んでから実行する。表情名は `character.yaml` の `expressions` のキー。

In [ ]:
paths = forge.inpaint("ally05", "hurt", source="/content/drive/MyDrive/illust-asset-forge/runs/<run_id>/<採用した顔>.png", n=4, on_image=F.show_one)

## 5. 全身（参考資料）

In [ ]:
paths = forge.generate("ally05", "fullbody", n=4, on_image=F.show_one)

## 試すときの書き方

- 足すタグ・消すタグ: `forge.generate("ally05", "face", extra="scar on cheek", remove="stubble")`（ネガティブは `extra_negative=`・`remove_negative=`）。`remove` は設定のタグにだけ効く。`extra` に設定と同じタグを重み付きで書くと、設定のタグが置き換わる（例: `extra="(stubble:1.3)"`）
- シードを固定: `seeds=[123, 456]`（渡すと `n` は無視される）
- guidance・steps（inpaint は `strength` も）: `guidance=6`、`steps=30`
- 画像を並べて見る: `F.show(paths)`
- **YAML は Colab で書き換えない**（次の pull でぶつかる）。採用した PNG を zip にして Claude に渡す。上書きの内容は PNG の条件（`overrides`）に残るので、Claude が設定に戻す。